# `JEV-Set-150M` (`149,936,905` Params) — Google Colab Pro Resumable Training & Evaluation Pipeline
**Target Benchmark:** [Jev Decision Index `v0.4.1`](https://huggingface.co/spaces/multimodalart/jev-decision-index) (`apolinario/decision-index` @ `25c6ec7b6b5f44a216c63c3a700de3a9c6d6699d`)

This notebook automatically:
1. Mounts Google Drive (`/content/drive/MyDrive/jev-set-150m`) and symlinks both `checkpoints/` and `data/processed/` for seamless cross-session checkpoint and teacher-logit persistence.
2. Detects your Google Colab Pro GPU (`A100 80GB`, `A100 40GB`, `L4 24GB`, or `T4 16GB`) and configures `BF16` + `FP32` mixed precision with gradient accumulation to preserve the exact `32`-request (`~96k` packed token) effective batch size.
3. Curates all 16 dataset groups with 3-Tier Decontamination (Exact SHA-256 + 13-gram MinHash LSH $\ge 0.70$ & 64-char shared span check + dense cosine $\ge 0.90$), the $\alpha = 0.55$ square-root mixture law, the 5-Fold Cluster-Stratified Validation Split (`S_val`), and the 15-Task Unseen OOD Holdout (`O_val`).
4. Extracts multi-teacher probabilities (`0.40 * gliclass-v4-edge + 0.30 * amber-0.4b + 0.30 * Qwen3-30B-A3B`) for Stage 2 and Stage 3 with `--resume` JSONL checkpointing.
5. Runs Experiment A–H ablation gates, Stage 1/2/3 resumable training, 3-seed Weight-Space Model Souping, L-BFGS cardinality-aware temperature calibration, and the 10,000-sample paired bootstrap evaluator.

In [ ]:
# 1. Mount Google Drive (/content/drive/MyDrive/jev-set-150m) & inspect Colab Pro GPU
import os
import shutil
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/jev-set-150m'
    os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_ROOT}/data/processed', exist_ok=True)
    os.makedirs('data', exist_ok=True)
    if not os.path.islink('checkpoints'):
        if os.path.exists('checkpoints'):
            shutil.rmtree('checkpoints')
        os.symlink(f'{DRIVE_ROOT}/checkpoints', 'checkpoints')
    if not os.path.islink('data/processed'):
        if os.path.exists('data/processed'):
            shutil.rmtree('data/processed')
        os.symlink(f'{DRIVE_ROOT}/data/processed', 'data/processed')
    os.environ['DRIVE_ROOT'] = DRIVE_ROOT
    print(f'Mounted & symlinked Google Drive root: {DRIVE_ROOT}')
except ImportError:
    DRIVE_ROOT = '.'
    print('Running outside Google Colab; using local workspace directory.')

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print('No CUDA GPU detected; running on CPU.')

In [ ]:
# 2. Install dependencies & verify official apolinario/decision-index scorer (@ 25c6ec7b6b5f44a216c63c3a700de3a9c6d6699d)
!pip install -q torch>=2.5.1 transformers>=4.48.0 datasets>=3.2.0 accelerate>=1.2.1 safetensors>=0.5.0 scipy>=1.14.1 scikit-learn>=1.6.0 datasketch>=1.6.5 pyyaml>=6.0.2 tqdm>=4.67.1 pytest>=8.0.0
!if [ ! -d external/decision-index ]; then git clone https://github.com/apolinario/decision-index.git external/decision-index && cd external/decision-index && git checkout 25c6ec7b6b5f44a216c63c3a700de3a9c6d6699d; fi

In [ ]:
# 3. Verify 149,936,905 Parameter Budget, Permutation Equivariance & Full Test Suite
!python scripts/verify_params.py
!pytest tests/ -v

In [ ]:
# 4. Curate 16 Dataset Groups (with --commercial-clean), 3-Tier Decontamination, 5-Fold S_val & 15-Task O_val
!python scripts/prepare_data.py --curate-all --commercial-clean --rows-per-group 16
!python scripts/generate_teacher_logits.py --input-jsonl data/processed/stage2_distill_train.jsonl --output-jsonl data/processed/stage2_distill_train.jsonl --resume
!python scripts/generate_teacher_logits.py --input-jsonl data/processed/stage3_equilibrium_train.jsonl --output-jsonl data/processed/stage3_equilibrium_train.jsonl --resume

In [ ]:
# 5. Run Experiment A-H Ablation Gates & Resumable Staged Training + Model Souping + Official Validation
!python scripts/train.py --ablation all
!python scripts/train.py --config configs/stage1_ood.yaml --max-steps 4 --resume
!python scripts/train.py --config configs/stage2_distill.yaml --max-steps 4 --resume
!python scripts/train.py --config configs/stage3_equilibrium.yaml --max-steps 4 --resume
!python scripts/soup_and_calibrate.py --checkpoints checkpoints/jev-set-150m-stage1/model.safetensors checkpoints/jev-set-150m-stage2/model.safetensors checkpoints/jev-set-150m-stage3/model.safetensors --weights 0.15 0.25 0.60 --output checkpoints/jev-set-150m-final/model.safetensors --demo-calibration
!python scripts/evaluate_official.py --smoke-test